# 05 ML GNN Embeddings

This notebook uses `MLTrainAndStore` to train regressors with only the GraphSAGE embedding features.

In [29]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
import xgboost as xgb
from xgboost import XGBRegressor

from src.models.ml_train_and_store import (
    MLTrainAndStore,
    load_gnn_ml_dataset,
    make_log_regression_model,
)

pd.set_option("display.max_columns", 200)

## Load Dataset

In [30]:
df, feature_cols = load_gnn_ml_dataset(PROJECT_ROOT)
df.shape, len(feature_cols)

((145536, 69), 64)

In [31]:
trainer = MLTrainAndStore(
    df=df,
    feature_cols=feature_cols,
    target_col="systemic_risk_label",
)

trainer.train_df.shape, trainer.val_df.shape, trainer.test_df.shape

((109152, 69), (18192, 69), (13644, 69))

## Define Models

In [32]:
candidate_models = {
    "linear_regression": make_log_regression_model(LinearRegression(), scale_features=True),
}

list(candidate_models)

['linear_regression']

## Train And Store

In [33]:
trainer.train_many(candidate_models)

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,linear_regression,0.18198,0.345751,0.274737,1.74918,3.086476,2.028126,0.120856,-0.024134,-0.05023


In [34]:
trainer.results()

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,linear_regression,0.18198,0.345751,0.274737,1.74918,3.086476,2.028126,0.120856,-0.024134,-0.05023


## Single-Model Pattern

In [36]:
amodel = make_log_regression_model(
    XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42),
    scale_features=True,
)

trainer.train_and_store(model=amodel, name="XGBRegressor_search_1")
trainer.results()

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,rf_search_1,0.057732,0.244163,0.167958,0.799158,3.02799,1.956651,0.816491,0.014311,0.02249
1,linear_regression,0.18198,0.345751,0.274737,1.74918,3.086476,2.028126,0.120856,-0.024134,-0.05023


## Best Model

In [ ]:
trainer.best_model_name()

In [ ]:
trainer.predict_test().head(20)